# Public source archive

Recovered historical code, prepared for public review. AI assisted the original coding and this archive preparation; team projects retain team attribution in the project README.

Outputs, notebook attachments, personal metadata, and embedded organizer datasets have been removed. Narrative cells retain headings only. Supply authorized input files before running data-dependent notebooks. This archive does not claim that manual checks were completed or that the code was independently developed.


# 老师starter修订版｜全部评价保留与分层比较
### 怎么用


## 第0步｜自动准备数据


In [ ]:
#@title 自动准备三家公司数据（通常无需展开）
from pathlib import Path
import os
if '_ALL_REVIEWS_ROOT' not in globals():
    _ALL_REVIEWS_ROOT=(Path('/content') if Path('/content').is_dir() else Path.cwd())/'Celonis_All_Reviews_Workspace'
_ALL_REVIEWS_ROOT.mkdir(parents=True,exist_ok=True)
os.chdir(_ALL_REVIEWS_ROOT)
# Public archive: original embedded organizer data removed.
from pathlib import Path
input_dir = Path('league_of_page_data')
required_files = ['organizations.csv', 'job_postings.csv', 'reviews.csv',
                  'review_job_role_links.csv', 'data_dictionary.csv']
missing_files = [name for name in required_files if not (input_dir/name).is_file()]
if missing_files:
    raise FileNotFoundError(
        'Place your authorized source CSVs in ' + str(input_dir.resolve())
        + '. Missing: ' + ', '.join(missing_files)
    )


## 第1步｜读取四张原始表，选三家公司


In [ ]:
# 1. 路径与读取：沿用老师starter的csv.DictReader；不依赖旧变量r/j
from pathlib import Path
import csv, json, re, html
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import display, HTML
except ImportError:
    class HTML:
        def __init__(self,data): self.data=data
    def display(value):
        if not isinstance(value,HTML): print(value.to_string(index=False) if hasattr(value,'to_string') else value)

NAMES=['Celonis','Anaplan','AlphaSense']
# 独立Notebook自带数据；复制到原starter时，也支持根目录原始CSV及01-等前缀。
DATA=Path('league_of_page_data') if Path('league_of_page_data').is_dir() else Path('.')
def load(name):
    matches=[p for p in [DATA/name,*sorted(DATA.glob('*-'+name))] if p.is_file()]
    if len(matches)!=1:
        raise FileNotFoundError(f'{name}: 请确认数据目录中恰有一份对应CSV，当前找到{len(matches)}份。')
    with matches[0].open(encoding='utf-8-sig',newline='') as f:
        return list(csv.DictReader(f))
organizations=pd.DataFrame(load('organizations.csv'))
source_reviews=pd.DataFrame(load('reviews.csv'))
source_jobs=pd.DataFrame(load('job_postings.csv'))
source_links=pd.DataFrame(load('review_job_role_links.csv'))
selected=organizations[organizations.organization_name.isin(NAMES)].copy()
assert len(selected)==3 and selected.organization_id.is_unique
ids=set(selected.organization_id)
all_reviews=source_reviews[source_reviews.organization_id.isin(ids)].copy()
all_jobs=source_jobs[source_jobs.organization_id.isin(ids)].copy()
all_links=source_links[source_links.organization_id.isin(ids)].copy()
lookup=selected.set_index('organization_id').organization_name
for frame in [all_reviews,all_jobs,all_links]:
    frame['company']=frame.organization_id.map(lookup)
assert all_reviews.review_id.is_unique and all_jobs.job_id.is_unique
assert set(all_links.review_id)<=set(all_reviews.review_id)
assert set(all_links.job_id)<=set(all_jobs.job_id)
original_review_ids=set(all_reviews.review_id)



## 第2步｜清理副本，添加分层标签


In [ ]:
# 2. 清理副本与分层标签：Senior、管理层、缺少标题/地点的评价全部保留
SENIOR=r'\bsenior\b|\bsr\.?\b|\bmanager\b|\bdirector\b|\bhead\b|\blead\b|\bvp\b|vice president|\bchief\b'
EARLY=r'\bintern\w*\b|\bjunior\b|\bgraduate\w*\b|\bentry[ -]level\b|\bworking student\b|\btrainee\b'
SALES=r'\bSDR\b|\bBDR\b|sales development|business development representative'
def clean(series):
    return series.fillna('').astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
for frame,title,date,textcols in [
    (all_reviews,'reviewer_job_title','review_date',['summary','pros','cons','advice_to_management']),
    (all_jobs,'title','posted_at',['description'])]:
    frame['title_clean']=clean(frame[title])
    frame['date_utc']=pd.to_datetime(frame[date],errors='coerce',utc=True)
    frame['senior_or_management']=frame.title_clean.str.contains(SENIOR,case=False,regex=True)
    frame['early_title']=frame.title_clean.str.contains(EARLY,case=False,regex=True)
    frame['sales_role']=frame.title_clean.str.contains(SALES,case=False,regex=True)
    frame['missing_title']=frame.title_clean.eq('')
    frame['text_clean']=frame[textcols].fillna('').astype(str).agg('\n'.join,axis=1)
    frame['missing_text']=frame.text_clean.str.strip().eq('')
all_reviews['missing_location']=clean(all_reviews.reviewer_location).eq('')
# 标记可能的面试语境，绝不按这个词直接删除
all_reviews['interview_candidate']=all_reviews.text_clean.str.contains(r'\binterview\w*\b|\bapplication process\b',case=False,regex=True)
all_reviews['duplicate_text']=all_reviews.duplicated(['organization_id','text_sha256'],keep=False)&all_reviews.text_sha256.ne('')
all_jobs['duplicate_text']=all_jobs.duplicated(['organization_id','text_sha256'],keep=False)&all_jobs.text_sha256.ne('')
# 公司各自对应广告城市；不把所有城市混合匹配给每家公司
city_patterns={'Celonis':r'New York|Raleigh','Anaplan':r'Minneapolis','AlphaSense':r'New York'}
all_reviews['advertised_city']=False
for name,pat in city_patterns.items():
    m=all_reviews.company.eq(name)
    all_reviews.loc[m,'advertised_city']=all_reviews.loc[m,'reviewer_location'].str.contains(pat,case=False,regex=True,na=False)
summary=[]
for name in NAMES:
    a=all_reviews[all_reviews.company.eq(name)]; b=all_jobs[all_jobs.company.eq(name)]
    summary.append({'公司':name,'全部广告':len(b),'全部评价':len(a),'初级标题候选':int(a.early_title.sum()),'资深或管理标题候选':int(a.senior_or_management.sum()),'销售开发相关评价':int(a.sales_role.sum()),'标题缺失':int(a.missing_title.sum()),'地点缺失':int(a.missing_location.sum())})
summary=pd.DataFrame(summary)
display(summary)



## 第3步｜先建立广告阶段基线


In [ ]:
# 3. 广告先行：共享关键词规则只定位主题，不自动确认公司支持或承诺
THEMES={
 'career':r'\blearn\w*\b|\btrain\w*\b|\bmentor\w*\b|\bonboard\w*\b|\bcoach\w*\b|career|promotion|progression',
 'work':r'work.life balance|work/life balance|workload|\bhybrid\b|\bremote\b|\bovertime\b|working hours|\bstress\w*\b|\bquota\w*\b',
 'benefits':r'\bbenefit\w*\b|\bcompensation\b|\bsalary\b|\bpay\b|\bcommission\w*\b|paid leave|\bPTO\b|health insurance'
}
LABELS={'career':'职业发展相关表达','work':'工作安排/负荷相关表达','benefits':'薪酬福利相关表达'}
for frame in [all_jobs,all_reviews]:
    for theme,pattern in THEMES.items():
        frame['candidate_'+theme]=frame.text_clean.str.contains(pattern,case=False,regex=True)
posting_rows=[]
for name in NAMES:
    group=all_jobs[all_jobs.company.eq(name)]
    for theme in THEMES:
        known=group[~group.missing_text]
        posting_rows.append({'公司':name,'主题':LABELS[theme],'源广告数':len(group),'非空文本广告数':len(known),'命中广告数':int(known['candidate_'+theme].sum()),'命中比例%':100*known['candidate_'+theme].mean() if len(known) else np.nan})
posting_screen=pd.DataFrame(posting_rows)

def show_table(title,frame):
    content=frame.to_html(index=False,escape=True)
    display(HTML('<details style="margin:10px 0;padding:12px;border:1px solid #aaa"><summary><b>'+html.escape(title)+'</b></summary><div style="overflow:auto;max-height:460px">'+content+'</div></details>'))
show_table('广告主题检索结果：不是承诺兑现率',posting_screen.round(2))
# 相近广告候选不排除Senior；保留标签供分层
sales_jobs=all_jobs[all_jobs.title_clean.str.contains(r'\bSDR\b|\bBDR\b|sales development|business development',case=False,regex=True)].copy()
show_table('销售开发广告候选：包括资深/管理标题，并保留层级标签',sales_jobs[['company','job_id','title','location_text','posted_at','early_title','senior_or_management']])



## 第4步｜清理评分，保留所有源评价


In [ ]:
# 4. 评分清理：只对1–5评分求均值；不删除评价；原始subratings保留
DIMS={'career_opportunities':'Career opportunities','work_life_balance':'Work-life balance','compensation_and_benefits':'Compensation & benefits'}
quality=[]
parsed=[]
for _,row in all_reviews.iterrows():
    try:
        raw=json.loads(row.subratings or '{}')
        if not isinstance(raw,dict): raise ValueError('subratings is not a JSON object')
    except (ValueError,TypeError):
        raw={}; quality.append({'review_id':row.review_id,'field':'subratings','value':row.subratings,'issue':'invalid JSON/object'})
    scores={}
    for key in DIMS:
        value=raw.get(key,'')
        try: number=float(value)
        except (ValueError,TypeError): number=np.nan
        valid=np.isfinite(number) and 1<=number<=5
        scores[key+'_valid']=number if valid else np.nan
        if not valid and value not in ['',None]:
            quality.append({'review_id':row.review_id,'field':key,'value':str(value),'issue':'missing/unavailable or outside 1–5; not counted as zero'})
    parsed.append(scores)
all_reviews=pd.concat([all_reviews.reset_index(drop=True),pd.DataFrame(parsed)],axis=1)
quality_log=pd.DataFrame(quality,columns=['review_id','field','value','issue'])
assert set(all_reviews.review_id)==original_review_ids



## 第5步｜主分析与分层敏感性


In [ ]:
# 5. 全部评价为主分析；子样本只用于稳健性检查，允许相互重叠
index=all_reviews.index
link_ids=set(all_links.review_id)
exact=all_links.location_match.eq('exact_location_text')
within=all_links.within_180_days.str.lower().eq('true')
linked180=set(all_links.loc[within,'review_id'])
linkedloc=set(all_links.loc[exact,'review_id'])
linkedboth=set(all_links.loc[exact&within,'review_id'])
SAMPLES={
 'All reviews':pd.Series(True,index=index),
 'Early-title candidates':all_reviews.early_title,
 'Sales roles incl. senior':all_reviews.sales_role,
 'Sales roles, no senior title':all_reviews.sales_role&~all_reviews.senior_or_management,
 'Senior/management titles':all_reviews.senior_or_management,
 'Sales + advertised city':all_reviews.sales_role&all_reviews.advertised_city,
 '2025 onward':all_reviews.date_utc.ge(pd.Timestamp('2025-01-01',tz='UTC')),
 'Same-title linked reviews':all_reviews.review_id.isin(link_ids),
 'Linked within 180 days':all_reviews.review_id.isin(linked180),
 'Linked exact location':all_reviews.review_id.isin(linkedloc),
 'Linked location + 180 days':all_reviews.review_id.isin(linkedboth)
}
score_rows=[]
for sample,mask in SAMPLES.items():
    for name in NAMES:
        group=all_reviews[mask&all_reviews.company.eq(name)]
        for key,label in DIMS.items():
            values=group[key+'_valid'].dropna()
            score_rows.append({'sample':sample,'company':name,'dimension':label,'n_reviews':len(group),'n_valid':len(values),'mean':values.mean() if len(values) else np.nan})
scores=pd.DataFrame(score_rows)
main_scores=scores[scores['sample'].eq('All reviews')].copy()
show_table('全部评价主分析：每个维度单独报告有效n',main_scores.round(3))
show_table('所有分层和关联敏感性：n=0表示证据不足，不是0分',scores.round(3))
show_table('评分异常/缺失日志：每行是一个字段问题',quality_log)



## 第6步｜显示三张核心图


In [ ]:
# 6. 三张核心图：全部样本、全部评价评分、Celonis样本敏感性
COLORS={'Celonis':'#6954d8','Anaplan':'#26968f','AlphaSense':'#e6a145'}
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10})
fig,ax=plt.subplots(figsize=(8,3.8));x=np.arange(3);w=.35
jobcounts=summary.set_index('公司').loc[NAMES,'全部广告']
revcounts=summary.set_index('公司').loc[NAMES,'全部评价']
ax.bar(x-w/2,jobcounts,w,label='Postings',color='#6954d8')
ax.bar(x+w/2,revcounts,w,label='All reviews',color='#26968f')
for i in range(3):
    ax.text(i-w/2,jobcounts.iloc[i]+5,str(jobcounts.iloc[i]),ha='center')
    ax.text(i+w/2,revcounts.iloc[i]+5,str(revcounts.iloc[i]),ha='center')
ax.set(xticks=x,xticklabels=NAMES,ylabel='Source records',ylim=(0,410),title='1 | All selected records retained');ax.legend();fig.tight_layout();plt.show()
fig,axes=plt.subplots(1,3,figsize=(12,4.5),sharey=True)
for ax,dimension in zip(axes,DIMS.values()):
    rows=main_scores[main_scores.dimension.eq(dimension)].set_index('company').reindex(NAMES)
    for i,name in enumerate(NAMES):
        row=rows.loc[name]
        if row.n_valid:
            ax.scatter(i,row['mean'],color=COLORS[name],s=70)
            ax.text(i,row['mean']+.13,f"{row['mean']:.2f}\nn={int(row.n_valid)}",ha='center',fontsize=9)
    ax.set(xticks=range(3),xticklabels=NAMES,ylim=(1,5.2),title=dimension)
    ax.tick_params(axis='x',rotation=25);ax.grid(axis='y',alpha=.15)
axes[0].set_ylabel('Mean valid 1–5 score')
fig.suptitle('2 | All reviews, including senior/management — descriptive only');fig.tight_layout();plt.show()
fig,ax=plt.subplots(figsize=(11,4.5))
shown=['All reviews','Early-title candidates','Sales roles incl. senior','Sales roles, no senior title','Senior/management titles','2025 onward']
for k,sample in enumerate(shown):
    g=scores[scores.company.eq('Celonis')&scores['sample'].eq(sample)].set_index('dimension').reindex(DIMS.values())
    xx=np.arange(3)+(k-(len(shown)-1)/2)*.12
    ax.scatter(xx,g['mean'],label=sample,s=40)
    for pos,(_,row) in zip(xx,g.iterrows()):
        if row.n_valid:ax.annotate(f"n={int(row.n_valid)}",(pos,row['mean']),xytext=(0,6),textcoords='offset points',ha='center',fontsize=7,rotation=50)
ax.set(xticks=range(3),xticklabels=list(DIMS.values()),ylim=(1,5.3),ylabel='Mean valid 1–5 score',title='3 | Celonis: overlapping sample sensitivity; small n needs caution')
ax.legend(fontsize=8,loc='lower left',ncol=2);ax.grid(axis='y',alpha=.15);fig.tight_layout();plt.show()



## 第7步｜全部原文在Colab直接查看


In [ ]:
# 7. 可展开全部933条原文：只读取，不把关键词当情绪或已验证事实
# 显示全部原文，可直接使用浏览器查找公司/ID；子样本不替代全部评价。
def passage(row,source):
    title=row.title_clean; rid=row.review_id if source=='review' else row.job_id
    date=row.review_date if source=='review' else row.posted_at
    location=row.reviewer_location if source=='review' else row.location_text
    heading=f'{row.company} | {rid} | {title or "[missing title]"} | {date}'
    if source=='review':
        body='\n\n'.join(f'{field}: {row[field]}' for field in ['summary','pros','cons','advice_to_management'])
    else:body=row.description
    return '<details style="margin:8px 0;padding:9px;border:1px solid #aaa"><summary>'+html.escape(heading)+'</summary><p>'+html.escape(location)+'</p><pre style="white-space:pre-wrap;font:14px/1.6 sans-serif">'+html.escape(body)+'</pre></details>'
_review_html=''
for name in NAMES:
    group=all_reviews[all_reviews.company.eq(name)].sort_values(['review_date','review_id'])
    _review_html+='<details style="margin:14px 0"><summary><b>'+html.escape(name)+f'：全部{len(group)}条评价（包含资深/管理及缺少元数据记录）</b></summary>'+''.join(passage(row,'review') for _,row in group.iterrows())+'</details>'
display(HTML(_review_html))
show_table('可能涉及面试的记录：仅标记，未删除；内部晋升也可能命中',all_reviews.loc[all_reviews.interview_candidate,['company','review_id','reviewer_job_title','review_date']])



## 第8步｜生成30段待人工核查样本


In [ ]:
# 8. 固定30条人工核查样本：广告/评价各15条，每家公司各5条
# 主题检索是否命中属于候选标签；人工标签默认空，不伪造完成情况。
audit_parts=[]
for source,frame,idcol in [('job',all_jobs,'job_id'),('review',all_reviews,'review_id')]:
    for name in NAMES:
        group=frame[frame.company.eq(name)].sort_values(idcol)
        take=group.sample(n=min(5,len(group)),random_state=20261023)
        for _,row in take.iterrows():
            rec={'source':source,'source_id':row[idcol],'company':name,'title':row.title_clean,'full_text':row.text_clean,'human_reviewer':'','human_notes':''}
            for theme in THEMES:
                rec[theme+'_candidate']=int(row['candidate_'+theme]);rec[theme+'_human']=None
            audit_parts.append(rec)
audit=pd.DataFrame(audit_parts)
# 阅读下方原文后，在这里填写；复合键避免广告ID与评价ID重名。
# 示例格式（请替换成实际核查记录，勿照抄示例）：
# HUMAN_LABELS={('review','实际ID'):{'reviewer':'你的姓名','career':1,'work':0,'benefits':0,'notes':'核查理由'}}
HUMAN_LABELS={}
for i,row in audit.iterrows():
    entry=HUMAN_LABELS.get((row.source,row.source_id))
    if not entry:continue
    if not str(entry.get('reviewer','')).strip():raise ValueError('人工核查必须填写reviewer')
    audit.loc[i,'human_reviewer']=entry['reviewer'];audit.loc[i,'human_notes']=entry.get('notes','')
    for theme in THEMES:
        value=entry.get(theme)
        if value is not None and value not in [0,1]:raise ValueError('人工主题标签只允许0、1或None')
        audit.loc[i,theme+'_human']=value
valid_audit_keys=set(zip(audit.source,audit.source_id))
assert set(HUMAN_LABELS)<=valid_audit_keys,'人工标签ID不在本次核查样本，请检查'
metrics=[]
for source in ['job','review']:
    for theme in THEMES:
        g=audit[audit.source.eq(source)&audit[theme+'_human'].notna()&audit.human_reviewer.ne('')]
        if len(g)==0:continue
        y=g[theme+'_human'].astype(int);p=g[theme+'_candidate'].astype(int)
        tp=int(((p==1)&(y==1)).sum());fp=int(((p==1)&(y==0)).sum());fn=int(((p==0)&(y==1)).sum())
        metrics.append({'source':source,'theme':theme,'n_checked':len(g),'agreement':(p==y).mean(),'precision':tp/(tp+fp) if tp+fp else np.nan,'recall':tp/(tp+fn) if tp+fn else np.nan})
_audit_html='<details><summary><b>30条待人工核查原文：点击展开</b></summary>'
for _,row in audit.iterrows():
    _audit_html+='<details style="margin:8px 0"><summary>'+html.escape(f'{row.source} {row.source_id} | {row.company} | {row.title}')+'</summary><pre style="white-space:pre-wrap">'+html.escape(row.full_text)+'</pre></details>'
_audit_html+='</details>'
display(HTML(_audit_html))
if metrics:show_table('已填写人工标签的检索验证结果：只适用于本次样本',pd.DataFrame(metrics).round(3))
else:print('人工核查尚未完成；不报告分类准确率。')
assert len(audit)==30 and len(all_reviews)==len(original_review_ids)
print('分析运行完成。全部评价保留；没有生成ZIP或自动提交。')


## 接下来如何回答题目
